# Plug-and-play ASR pipeline — milestone 7

This notebook is the complete thin smoke workflow for the pipeline built across milestones 1–6. It verifies data preparation, cache behavior, base prediction/evaluation, smoke training, tuned prediction/evaluation, and base-vs-tuned reporting.

It is intentionally smoke-safe: it does **not** download real models, run full training, or run expensive inference.

## 1. Imports

In [ ]:
from pathlib import Path
import sys
import traceback
from dataclasses import replace

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from asr_pipeline.config import load_config, save_resolved_config_json
from asr_pipeline.registry import get_model_family, create_adapter
from asr_pipeline.data import (
    clear_preparation_cache,
    generate_fake_smoke_dataset,
    prepare_data_and_collator,
)
from asr_pipeline.predict import clear_prediction_cache, predict
from asr_pipeline.evaluate import (
    clear_metric_cache,
    compare_base_vs_tuned,
    evaluate_predictions,
)
from asr_pipeline.train import clear_training_outputs, train
from asr_pipeline.utils.hashing import config_hash
from asr_pipeline.utils.io import ensure_dir, write_json

print(f"Project root: {project_root}")

## 2. Config setup

Load `configs/smoke_config.yaml`, resolve relative paths against the project root, and clear old smoke artifacts so first-run/second-run cache assertions are deterministic.

In [ ]:
config_path = project_root / "configs" / "smoke_config.yaml"
base_config = load_config(config_path)

base_config = replace(
    base_config,
    train_path=str(project_root / base_config.train_path),
    val_path=str(project_root / base_config.val_path),
    test_path=str(project_root / base_config.test_path),
    output_dir=str(project_root / base_config.output_dir),
    local_model_cache_dir=str(project_root / base_config.local_model_cache_dir),
    run_name="milestone7_smoke",
    smoke_mode=True,
)

reports_dir = ensure_dir(Path(base_config.output_dir) / "reports")
resolved_config_path = reports_dir / f"{base_config.run_name}_resolved_config.json"
save_resolved_config_json(base_config, resolved_config_path)

# Deterministic notebook verification: remove old smoke outputs before the first pass.
clear_preparation_cache(base_config)
clear_prediction_cache(base_config, tuned_or_base="base")
clear_prediction_cache(base_config, tuned_or_base="tuned")
clear_metric_cache(base_config, tuned_or_base="base")
clear_metric_cache(base_config, tuned_or_base="tuned")
clear_training_outputs(base_config)

for report_path in reports_dir.glob("milestone7_*.json"):
    report_path.unlink()
for report_path in reports_dir.glob("base_vs_tuned__*"):
    report_path.unlink()

print(f"Loaded config from: {config_path}")
print(f"Resolved config saved to: {resolved_config_path}")
print(f"Config hash: {config_hash(base_config)}")
print(f"Output dir: {base_config.output_dir}")
print(f"W&B mode: {base_config.wandb_mode}")

## 3. Smoke model list

Each listed model represents one architecture family. Larger models in the same family use the same adapter dispatch path.

In [ ]:
smoke_representative_models = [
    "openai/whisper-medium",
    "Qwen/Qwen3-ASR-0.6B",
    "Omni ASR 300M",
]

for model_name in smoke_representative_models:
    print(f"{model_name} -> {get_model_family(model_name)}")

## 4. Fake smoke data creation

Generate one tiny 16 kHz WAV + JSONL row for each split: train, validation, and test.

In [ ]:
fake_data = generate_fake_smoke_dataset(project_root / "data" / "smoke", overwrite=True)

print("Fake smoke data generated:")
for split, path in fake_data["split_paths"].items():
    print(f"  {split}: {path}")

print("\nDataset schema fields:")
print("uid, audio_path, text, duration, sample_rate, source")

## 5. Loop over smoke models

For each architecture, run the full smoke workflow:

1. instantiate adapter
2. prepare data twice and verify preparation cache
3. run base prediction twice and verify prediction cache
4. evaluate base prediction twice and verify metric cache
5. run smoke training
6. run tuned prediction
7. evaluate tuned prediction
8. compare base vs tuned
9. collect a summary row

Failures are caught per architecture and written under `outputs/reports/`.

In [ ]:
summary_rows = []
error_reports = []

summary_columns = [
    "model name",
    "architecture",
    "preparation first run status",
    "preparation second run cache status",
    "base prediction cache status",
    "base evaluation cache status",
    "training status",
    "LoRA backend",
    "best checkpoint",
    "base WER",
    "tuned WER",
    "WER improvement",
    "errors if any",
]

for model_name in smoke_representative_models:
    row = {column: "" for column in summary_columns}
    row["model name"] = model_name
    row["best checkpoint"] = ""
    row["base WER"] = None
    row["tuned WER"] = None
    row["WER improvement"] = None
    step = "start"

    try:
        family = get_model_family(model_name)
        row["architecture"] = family
        cfg = replace(base_config, model_name=model_name, run_name=f"milestone7_smoke_{family}")
        adapter = create_adapter(cfg)
        split_paths = {
            "train": cfg.train_path,
            "val": cfg.val_path,
            "test": cfg.test_path,
        }
        print(f"\n=== {model_name} ({family}) ===")
        print(f"Adapter: {adapter.__class__.__name__}")

        step = "prepare data first run"
        prep_first = prepare_data_and_collator(cfg, adapter, split_paths)
        assert prep_first["metadata"]["cache_created"] is True
        row["preparation first run status"] = "created cache"

        step = "prepare data second run/cache verification"
        prep_second = prepare_data_and_collator(cfg, adapter, split_paths)
        assert prep_second["metadata"]["cache_loaded"] is True
        row["preparation second run cache status"] = "loaded cache"
        prepared_data = prep_second
        collator = prep_second["collator"]
        print(f"Collator: {collator.__class__.__name__}")

        step = "base prediction first run"
        base_pred_first = predict(cfg, adapter, prepared_data, split="test")
        assert base_pred_first["metadata"]["predicted_new"] is True
        assert base_pred_first["metadata"]["loaded_from_cache"] is False

        step = "base prediction second run/cache verification"
        base_pred_second = predict(cfg, adapter, prepared_data, split="test")
        assert base_pred_second["metadata"]["loaded_from_cache"] is True
        row["base prediction cache status"] = "created, then loaded cache"

        step = "base evaluation first run"
        base_eval_first = evaluate_predictions(cfg, base_pred_second["metadata"]["path"])
        assert base_eval_first["metadata"]["computed_new"] is True
        assert base_eval_first["metadata"]["loaded_from_cache"] is False

        step = "base evaluation second run/cache verification"
        base_eval_second = evaluate_predictions(cfg, base_pred_second["metadata"]["path"])
        assert base_eval_second["metadata"]["loaded_from_cache"] is True
        row["base evaluation cache status"] = "computed, then loaded cache"
        row["base WER"] = base_eval_second["metrics"]["wer"]

        step = "smoke training"
        train_result = train(cfg, adapter, prepared_data, collator)
        train_meta = train_result["metadata"]
        training = train_result["training"]
        assert train_meta["train_status"] == "success"
        best_checkpoint = train_meta["best_checkpoint_path"]
        assert Path(best_checkpoint).exists()
        row["training status"] = train_meta["train_status"]
        row["LoRA backend"] = f"{train_meta['lora_backend']} ({train_meta['lora_backend_mode']})"
        row["best checkpoint"] = best_checkpoint

        step = "tuned prediction"
        tuned_pred = predict(cfg, adapter, prepared_data, split="test", tuned_adapter_path=best_checkpoint)
        assert Path(tuned_pred["metadata"]["path"]).exists()
        assert tuned_pred["metadata"]["tuned_or_base"] == "tuned"

        step = "tuned evaluation"
        tuned_eval = evaluate_predictions(cfg, tuned_pred["metadata"]["path"])
        assert Path(tuned_eval["metadata"]["metrics_path"]).exists()
        row["tuned WER"] = tuned_eval["metrics"]["wer"]

        step = "base vs tuned comparison"
        comparison = compare_base_vs_tuned(
            base_eval_second["metadata"]["metrics_path"],
            tuned_eval["metadata"]["metrics_path"],
        )
        assert Path(comparison["metadata"]["comparison_json_path"]).exists()
        assert Path(comparison["metadata"]["comparison_markdown_path"]).exists()
        row["WER improvement"] = comparison["comparison"]["metrics"]["wer"]["absolute_improvement"]
        row["errors if any"] = ""

        # Keep useful output paths for later inspection without bloating the final table.
        row["_base_prediction_path"] = base_pred_second["metadata"]["path"]
        row["_base_metrics_path"] = base_eval_second["metadata"]["metrics_path"]
        row["_tuned_prediction_path"] = tuned_pred["metadata"]["path"]
        row["_tuned_metrics_path"] = tuned_eval["metadata"]["metrics_path"]
        row["_comparison_json_path"] = comparison["metadata"]["comparison_json_path"]
        row["_comparison_markdown_path"] = comparison["metadata"]["comparison_markdown_path"]
        row["_train_loss"] = training["last_metrics"]["train_loss"]
        row["_val_loss"] = training["last_metrics"]["val_loss"]
        row["_cer"] = training["last_metrics"]["cer"]
        row["_best_epoch"] = train_meta["best_epoch"]

        print("Status: passed full smoke workflow")
        print(f"Base WER: {row['base WER']:.4f} | Tuned WER: {row['tuned WER']:.4f} | Improvement: {row['WER improvement']:.4f}")

    except Exception as exc:
        error_payload = {
            "model_name": model_name,
            "architecture": row.get("architecture", "unknown"),
            "failed_step": step,
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        }
        safe_model_name = model_name.replace("/", "__").replace(" ", "_").replace(":", "_")
        error_path = reports_dir / f"milestone7_error__{safe_model_name}.json"
        write_json(error_path, error_payload)
        error_payload["error_report_path"] = str(error_path)
        error_reports.append(error_payload)
        row["errors if any"] = f"{step}: {repr(exc)}"
        print(f"ERROR for {model_name} at step '{step}': {exc!r}")
        print(f"Saved detailed error report: {error_path}")

    finally:
        summary_rows.append(row)

aggregate_error_path = reports_dir / "milestone7_error_report.json"
write_json(aggregate_error_path, error_reports)
print(f"\nAggregate error report saved to: {aggregate_error_path}")
print(f"Architectures attempted: {len(smoke_representative_models)}")
print(f"Architectures with errors: {len(error_reports)}")

## 6. Final summary table

In [ ]:
def _fmt(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    text = str(value)
    if len(text) > 120:
        return text[:117] + "..."
    return text

def print_markdown_table(rows, columns):
    print("| " + " | ".join(columns) + " |")
    print("|" + "|".join(["---"] * len(columns)) + "|")
    for row in rows:
        print("| " + " | ".join(_fmt(row.get(column, "")).replace("|", "\\|") for column in columns) + " |")

print_markdown_table(summary_rows, summary_columns)

summary_path = reports_dir / "milestone7_summary.json"
write_json(summary_path, summary_rows)
print(f"\nSummary JSON saved to: {summary_path}")

## 7. Error report summary

In [ ]:
if error_reports:
    print("One or more architectures failed. The notebook continued and saved detailed reports:")
    for err in error_reports:
        print(f"- {err['model_name']} failed at {err['failed_step']}: {err['error']}")
        print(f"  report: {err['error_report_path']}")
else:
    print("No architecture errors were reported. All smoke representative models completed the end-to-end workflow.")

print(f"Aggregate error report: {aggregate_error_path}")

## 8. Instructions for switching to real training

To move from smoke mode to real training:

1. Edit `configs/smoke_config.yaml` or create a new config file.
2. Set `smoke_mode: false`.
3. Point `train_path`, `val_path`, and `test_path` to real JSONL metadata files using the same schema: `uid`, `audio_path`, `text`, `duration`, `sample_rate`, `source`.
4. Keep `local_model_cache_dir` on persistent storage so model downloads are reused.
5. Choose `wandb_mode`:
   - `disabled` for no W&B logging
   - `offline` for local W&B logs
   - `online` for normal W&B logging
6. Replace the smoke placeholder branches inside the architecture adapters with real model loading, LoRA setup, Trainer/loop code, and decoding.
7. Keep using the same high-level APIs from this notebook:
   - `prepare_data_and_collator(...)`
   - `predict(...)`
   - `evaluate_predictions(...)`
   - `train(...)`
   - `compare_base_vs_tuned(...)`

Changing only `model_name` is enough to switch architecture dispatch, as long as the model name is registered in `asr_pipeline/registry.py`.